Dacon ?? ??? 1 ??? ?? ?? ??
Overfit
2020? 02? 03? (????)
??? ?? ????
A ?? ??

1) ???? ?? ?? ??. ?? ??? ?? ??? ???? ??? ??? ? ??? ?

2) ?? ??? ???? R user? R or .rmd. Python user? .py or .ipynb

3) ??? ?/data? ??? ?/?? ?? ?? ?? or R? ?? setwd(" "), python? ?? os.chdir? ???? ?? ??

4) ?? ????? ?????? ???? ??? ???? ??? ??? ??

5) ?? ??? ?? ?? ????? ?(????? ?? ?? ????? ?).

6) ??? ??? ???? ?? UTF-8? ????? ?

B ?? ??? ??

1) ?? ?? ??? (?? ?? ?) ??? ????, ?? ?? ? ?? ??

2) ?? ??? ?? ?? ???? ??? ??? ?? ???? ?? ??

3) ?? ???? ???? ??, ??? ??? ?? ??

1. ????? ? ???
Library & Data

In [ ]:
# ??? ??? ??? ??.

# OS : Ubuntu 18.04.3 LTS
# CPU : Intel(R) Xeon(R) CPU @ 2.30GHz (4 CPUs.)
# GPU : Nvidia Tesla T4 (1 GPU.)

# Python Version : 3.7.6
# Nvidia Driver Version : 440.33.01
# CUDA Version (PyTorch) : 10.1.243
# cuDNN Version (PyTorch) : 7.6.3

In [ ]:
import time
import random
import numpy as np  # 1.18.1
from numpy.random import shuffle
import pandas as pd  # 0.25.3
import torch  # 1.4.0
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset
from torch.utils.data.sampler import Sampler, SequentialSampler
from torch.backends import cudnn

#?? ??? ?? directory? 'train.csv'? 'test.csv'? ???? 'Data' ?? ???.

2. ??? ???
Data Cleansing & Pre-Processing

In [ ]:
# ??? ???? ?? ?????? ??? ?? ??.

3. ??? ????
Exploratory Data Analysis


In [ ]:
# pandas? ???? correlation matrix? ???? ????.
# (?? ??? ???? ??? ????? code? ???.)

4. ?? ?? ? ?? ??
Feature Engineering & Initial Modeling

In [ ]:
# fully-connected NN(layer 5? ??)? ??? ?? ???? under-fitting?? ??? ???.
# depth? ?? fully-connected NN? ????? ????? ?? ensemble?? ??? ???.
# (?? ??? ???? ??? ????? code? ???.)

5. ?? ?? ? ??
Model Tuning & Evaluation

In [ ]:
# ?? ??? ?? code ??? ??? ??.

# ContinuousBatchSampler? mini-batch? ??? ?? ???? ????? ?? BatchSampler?.
# ?? ??? ?? L1Loss? ???.
# SGD optimizer? ???.
# CosineAnnealingWarmRestarts? ???? learning rate? ???.

In [ ]:
# ?? 1

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class ContinuousBatchSampler(Sampler):
    def __init__(self, sampler, batch_size, drop_last):
        self.sampler = sampler
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.from_last_epoch = []

    def __iter__(self):
        idx_from_sampler = set(self.sampler)
        idx_to_exclude = set(self.from_last_epoch)
        idx_after_exclusion = sorted(list(idx_from_sampler - idx_to_exclude))
        shuffle(idx_after_exclusion)
        first_batch = self.from_last_epoch + idx_after_exclusion[:self.batch_size - len(self.from_last_epoch)]
        yield first_batch
        idx_of_left = sorted(idx_after_exclusion[self.batch_size - len(self.from_last_epoch):] + list(idx_to_exclude))
        shuffle(idx_of_left)
        batch = []
        for idx in idx_of_left:
            batch.append(idx)
            if len(batch) == self.batch_size:
                yield batch
                batch = []
        if not self.drop_last:
            self.from_last_epoch = batch.copy()

    def __len__(self):
        if self.drop_last:
            return len(self.sampler) // self.batch_size
        else:
            return (len(self.sampler) + len(self.from_last_epoch)) // self.batch_size


device = torch.device('cuda:0')
num_epochs = 1000
batch_size = 2048
initial_learning_rate = 0.2
loader_params = {'num_workers': 8, 'pin_memory': True}

train_data = np.array(pd.read_csv('./Data/train.csv'), dtype=np.float32)
X_train = torch.tensor(train_data[:, 4:], dtype=torch.float32)
y_train = torch.tensor(train_data[:, :4], dtype=torch.float32)

dataset = TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(dataset, batch_sampler=ContinuousBatchSampler(
    sampler=SequentialSampler(range(len(dataset))), batch_size=batch_size, drop_last=False), **loader_params)

prediction = np.zeros((10000, 4), dtype=np.float32)
logging_term = 1000
logging_total = int(810000 * num_epochs / batch_size)

for model_no in ['Model_01', 'Model_02', 'Model_03', 'Model_04', 'Model_05',
                 'Model_06', 'Model_07', 'Model_08', 'Model_09', 'Model_10',
                 'Model_11', 'Model_12', 'Model_13', 'Model_14', 'Model_15',
                 'Model_16', 'Model_17', 'Model_18', 'Model_19', 'Model_20']:

    net = nn.Sequential(
        nn.BatchNorm1d(226),
        nn.ReLU(),
        nn.Linear(226, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 4)
    )

    model = net.to(device)
    running_loss = 0.
    running_counter = 0

    criterion = torch.nn.L1Loss()
    optimizer = optim.SGD(model.parameters(), lr=initial_learning_rate, momentum=0.9)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=250, T_mult=1, eta_min=0.005,
                                                               last_epoch=-1)

    for epoch in range(num_epochs):

        model.train()
        for xx, yy in train_loader:

            xx, yy = xx.to(device), yy.to(device)
            with torch.no_grad():
                xx += torch.randn((xx.shape[0], 226), device='cuda:0') * 0.003

            out = model(xx)
            loss = criterion(out, yy)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_counter += 1
            running_loss += loss.item()

            if running_counter % logging_term == 0:
                print(model_no + ' (iter {:6d}/{:6d}) {:.4f}'.format(running_counter, logging_total,
                                                                     running_loss / logging_term))
                running_loss = 0.

        scheduler.step()

    model.eval()
    output = model(
        torch.tensor(np.array(pd.read_csv('./Data/test.csv'), dtype=np.float32))[:, 1:].to(device))
    output = np.array(output.detach().to('cpu'), dtype=np.float32)
    prediction += output * 0.05

prediction = pd.DataFrame({'id': np.array(list(range(10000)), dtype=np.int32),
                           'layer_1': prediction[:, 0],
                           'layer_2': prediction[:, 1],
                           'layer_3': prediction[:, 2],
                           'layer_4': prediction[:, 3]})
prediction.to_csv('./prediction_result_{}.csv'.format(time.strftime('%y%m%d%H%M%S')), index=False)

In [ ]:
# ?? 2

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class ContinuousBatchSampler(Sampler):
    def __init__(self, sampler, batch_size, drop_last):
        self.sampler = sampler
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.from_last_epoch = []

    def __iter__(self):
        idx_from_sampler = set(self.sampler)
        idx_to_exclude = set(self.from_last_epoch)
        idx_after_exclusion = sorted(list(idx_from_sampler - idx_to_exclude))
        shuffle(idx_after_exclusion)
        first_batch = self.from_last_epoch + idx_after_exclusion[:self.batch_size - len(self.from_last_epoch)]
        yield first_batch
        idx_of_left = sorted(idx_after_exclusion[self.batch_size - len(self.from_last_epoch):] + list(idx_to_exclude))
        shuffle(idx_of_left)
        batch = []
        for idx in idx_of_left:
            batch.append(idx)
            if len(batch) == self.batch_size:
                yield batch
                batch = []
        if not self.drop_last:
            self.from_last_epoch = batch.copy()

    def __len__(self):
        if self.drop_last:
            return len(self.sampler) // self.batch_size
        else:
            return (len(self.sampler) + len(self.from_last_epoch)) // self.batch_size


device = torch.device('cuda:0')
num_epochs = 1000
batch_size = 2048
initial_learning_rate = 0.16
loader_params = {'num_workers': 8, 'pin_memory': True}

train_data = np.array(pd.read_csv('./Data/train.csv'), dtype=np.float32)
X_train = torch.tensor(train_data[:, 4:], dtype=torch.float32)
y_train = torch.tensor(train_data[:, :4], dtype=torch.float32)

dataset = TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(dataset, batch_sampler=ContinuousBatchSampler(
    sampler=SequentialSampler(range(len(dataset))), batch_size=batch_size, drop_last=False), **loader_params)

prediction = np.zeros((10000, 4), dtype=np.float32)
logging_term = 1000
logging_total = int(810000 * num_epochs / batch_size)

for model_no in ['Model_01', 'Model_02', 'Model_03', 'Model_04', 'Model_05', 'Model_06', 'Model_07']:

    net = nn.Sequential(
        nn.Linear(226, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 4)
    )

    model = net.to(device)
    running_loss = 0.
    running_counter = 0

    criterion = torch.nn.L1Loss()
    optimizer = optim.SGD(model.parameters(), lr=initial_learning_rate, momentum=0.9, weight_decay=2e-6)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=250, T_mult=1, eta_min=0.005,
                                                               last_epoch=-1)

    for epoch in range(num_epochs):

        model.train()
        for xx, yy in train_loader:

            xx, yy = xx.to(device), yy.to(device)
            with torch.no_grad():
                xx += torch.randn((xx.shape[0], 226), device='cuda:0') * 0.00325

            out = model(xx)
            loss = criterion(out, yy)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_counter += 1
            running_loss += loss.item()

            if running_counter % logging_term == 0:
                print(model_no + ' (iter {:6d}/{:6d}) {:.4f}'.format(running_counter, logging_total,
                                                                     running_loss / logging_term))
                running_loss = 0.

        scheduler.step()

    model.eval()
    output = model(
        torch.tensor(np.array(pd.read_csv('./Data/test.csv'), dtype=np.float32))[:, 1:].to(device))
    output = np.array(output.detach().to('cpu'), dtype=np.float32)
    prediction += output * 1 / 7

prediction = pd.DataFrame({'id': np.array(list(range(10000)), dtype=np.int32),
                           'layer_1': prediction[:, 0],
                           'layer_2': prediction[:, 1],
                           'layer_3': prediction[:, 2],
                           'layer_4': prediction[:, 3]})
prediction.to_csv('./prediction_result_{}.csv'.format(time.strftime('%y%m%d%H%M%S')), index=False)

In [ ]:
# ?? 3

random.seed(1)
np.random.seed(1)
torch.manual_seed(1)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class ContinuousBatchSampler(Sampler):
    def __init__(self, sampler, batch_size, drop_last):
        self.sampler = sampler
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.from_last_epoch = []

    def __iter__(self):
        idx_from_sampler = set(self.sampler)
        idx_to_exclude = set(self.from_last_epoch)
        idx_after_exclusion = sorted(list(idx_from_sampler - idx_to_exclude))
        shuffle(idx_after_exclusion)
        first_batch = self.from_last_epoch + idx_after_exclusion[:self.batch_size - len(self.from_last_epoch)]
        yield first_batch
        idx_of_left = sorted(idx_after_exclusion[self.batch_size - len(self.from_last_epoch):] + list(idx_to_exclude))
        shuffle(idx_of_left)
        batch = []
        for idx in idx_of_left:
            batch.append(idx)
            if len(batch) == self.batch_size:
                yield batch
                batch = []
        if not self.drop_last:
            self.from_last_epoch = batch.copy()

    def __len__(self):
        if self.drop_last:
            return len(self.sampler) // self.batch_size
        else:
            return (len(self.sampler) + len(self.from_last_epoch)) // self.batch_size


device = torch.device('cuda:0')
num_epochs = 1000
batch_size = 2048
initial_learning_rate = 0.16
loader_params = {'num_workers': 8, 'pin_memory': True}

train_data = np.array(pd.read_csv('./Data/train.csv'), dtype=np.float32)
X_train = torch.tensor(train_data[:, 4:], dtype=torch.float32)
y_train = torch.tensor(train_data[:, :4], dtype=torch.float32)

dataset = TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(dataset, batch_sampler=ContinuousBatchSampler(
    sampler=SequentialSampler(range(len(dataset))), batch_size=batch_size, drop_last=False), **loader_params)

prediction = np.zeros((10000, 4), dtype=np.float32)
logging_term = 1000
logging_total = int(810000 * num_epochs / batch_size)

for model_no in ['Model_01', 'Model_02', 'Model_03', 'Model_04', 'Model_05', 'Model_06', 'Model_07']:

    net = nn.Sequential(
        nn.Linear(226, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 768),
        nn.BatchNorm1d(768),
        nn.ReLU(),
        nn.Linear(768, 4)
    )

    model = net.to(device)
    running_loss = 0.
    running_counter = 0

    criterion = torch.nn.L1Loss()
    optimizer = optim.SGD(model.parameters(), lr=initial_learning_rate, momentum=0.9, weight_decay=2e-6)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=250, T_mult=1, eta_min=0.005,
                                                               last_epoch=-1)

    for epoch in range(num_epochs):

        model.train()
        for xx, yy in train_loader:

            xx, yy = xx.to(device), yy.to(device)
            with torch.no_grad():
                xx += torch.randn((xx.shape[0], 226), device='cuda:0') * 0.00325

            out = model(xx)
            loss = criterion(out, yy)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_counter += 1
            running_loss += loss.item()

            if running_counter % logging_term == 0:
                print(model_no + ' (iter {:6d}/{:6d}) {:.4f}'.format(running_counter, logging_total,
                                                                     running_loss / logging_term))
                running_loss = 0.

        scheduler.step()

    model.eval()
    output = model(
        torch.tensor(np.array(pd.read_csv('./Data/test.csv'), dtype=np.float32))[:, 1:].to(device))
    output = np.array(output.detach().to('cpu'), dtype=np.float32)
    prediction += output * 1 / 7

prediction = pd.DataFrame({'id': np.array(list(range(10000)), dtype=np.int32),
                           'layer_1': prediction[:, 0],
                           'layer_2': prediction[:, 1],
                           'layer_3': prediction[:, 2],
                           'layer_4': prediction[:, 3]})
prediction.to_csv('./prediction_result_{}.csv'.format(time.strftime('%y%m%d%H%M%S')), index=False)

In [ ]:
# ?? 4

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class ContinuousBatchSampler(Sampler):
    def __init__(self, sampler, batch_size, drop_last):
        self.sampler = sampler
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.from_last_epoch = []

    def __iter__(self):
        idx_from_sampler = set(self.sampler)
        idx_to_exclude = set(self.from_last_epoch)
        idx_after_exclusion = sorted(list(idx_from_sampler - idx_to_exclude))
        shuffle(idx_after_exclusion)
        first_batch = self.from_last_epoch + idx_after_exclusion[:self.batch_size - len(self.from_last_epoch)]
        yield first_batch
        idx_of_left = sorted(idx_after_exclusion[self.batch_size - len(self.from_last_epoch):] + list(idx_to_exclude))
        shuffle(idx_of_left)
        batch = []
        for idx in idx_of_left:
            batch.append(idx)
            if len(batch) == self.batch_size:
                yield batch
                batch = []
        if not self.drop_last:
            self.from_last_epoch = batch.copy()

    def __len__(self):
        if self.drop_last:
            return len(self.sampler) // self.batch_size
        else:
            return (len(self.sampler) + len(self.from_last_epoch)) // self.batch_size


device = torch.device('cuda:0')
num_epochs = 1000
batch_size = 2048
initial_learning_rate = 0.16
loader_params = {'num_workers': 8, 'pin_memory': True}

train_data = np.array(pd.read_csv('./Data/train.csv'), dtype=np.float32)
X_train = torch.tensor(train_data[:, 4:], dtype=torch.float32)
y_train = torch.tensor(train_data[:, :4], dtype=torch.float32)

dataset = TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(dataset, batch_sampler=ContinuousBatchSampler(
    sampler=SequentialSampler(range(len(dataset))), batch_size=batch_size, drop_last=False), **loader_params)

prediction = np.zeros((10000, 4), dtype=np.float32)
logging_term = 1000
logging_total = int(810000 * num_epochs / batch_size)


class net(nn.Module):
    def __init__(self):
        super().__init__()
        self.avg = nn.AvgPool1d(kernel_size=3, stride=3, padding=1)
        self.linear = nn.Sequential(
            nn.Linear(76, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Linear(768, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Linear(768, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Linear(768, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Linear(768, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Linear(768, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Linear(768, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Linear(768, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(),
            nn.Linear(768, 4)
        )

    def forward(self, x):
        x = x.view(x.shape[0], 1, -1)
        x = self.avg(x)
        x = x.view(x.shape[0], -1)
        x = self.linear(x)
        return x


for model_no in ['Model_01', 'Model_02', 'Model_03', 'Model_04', 'Model_05', 'Model_06', 'Model_07']:

    model = net().to(device)
    running_loss = 0.
    running_counter = 0

    criterion = torch.nn.L1Loss()
    optimizer = optim.SGD(model.parameters(), lr=initial_learning_rate, momentum=0.9, weight_decay=2e-6)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=250, T_mult=1, eta_min=0.005,
                                                               last_epoch=-1)

    for epoch in range(num_epochs):

        model.train()
        for xx, yy in train_loader:

            xx, yy = xx.to(device), yy.to(device)
            with torch.no_grad():
                xx += torch.randn((xx.shape[0], 226), device='cuda:0') * 0.005

            out = model(xx)
            loss = criterion(out, yy)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_counter += 1
            running_loss += loss.item()

            if running_counter % logging_term == 0:
                print(model_no + ' (iter {:6d}/{:6d}) {:.4f}'.format(running_counter, logging_total,
                                                                     running_loss / logging_term))
                running_loss = 0.

        scheduler.step()

    model.eval()
    output = model(
        torch.tensor(np.array(pd.read_csv('./Data/test.csv'), dtype=np.float32))[:, 1:].to(device))
    output = np.array(output.detach().to('cpu'), dtype=np.float32)
    prediction += output * 1 / 7

prediction = pd.DataFrame({'id': np.array(list(range(10000)), dtype=np.int32),
                           'layer_1': prediction[:, 0],
                           'layer_2': prediction[:, 1],
                           'layer_3': prediction[:, 2],
                           'layer_4': prediction[:, 3]})
prediction.to_csv('./prediction_result_{}.csv'.format(time.strftime('%y%m%d%H%M%S')), index=False)

In [ ]:
# ?? 4?? ?? ??? ???? 4?? ?? ?? ??? ???.
# ?? ?? ??? ??? ensemble?? ??? ???.
# ??? ??? ??? ??? ?? ???.

file_list = np.sort(glob('./*.csv'))

file_0 = pd.read_csv(file_list[0]).to_numpy()[:, 1:]
file_1 = pd.read_csv(file_list[1]).to_numpy()[:, 1:]
file_2 = pd.read_csv(file_list[2]).to_numpy()[:, 1:]
file_3 = pd.read_csv(file_list[3]).to_numpy()[:, 1:]

prediction = (file_0 + file_1 + file_2 + file_3) / 4

prediction = pd.DataFrame({'id': np.array(list(range(10000)), dtype=np.int32),
                           'layer_1': prediction[:, 0],
                           'layer_2': prediction[:, 1],
                           'layer_3': prediction[:, 2],
                           'layer_4': prediction[:, 3]})
prediction.to_csv('../prediction_result_{}.csv'.format(time.strftime('%y%m%d%H%M%S')), index=False)

6. ?? ? ??
Conclusion & Discussion

In [ ]:
# ?? ?? model? ???? ensemble??? ???(stochastically)?? 
# under-fitting ??? ???? ???? model? ??? ? ???.

# ?? ??? ?? ?? ??? ????? ???? model?(depth, width ?)?
# ? ?? ???? ??? ???? under-fitting ??? ??? ? ?? ??? ???.
# ??, ?? ???? ?? AutoML ?? ??? ???? ?? ??? ? ??.